## 1 - Validação e Tratamento de Dados Ausentes


#### 1.1) - Fiz uma cópia do .csv na pasta ‘C:\Dev\sctec\sctec_mini-projeto_avaliativo’
#### 1.2) - Ler `halloween_vendas.csv` com `pd.read_csv()`, da biblioteca Pandas.

In [3]:
import pandas as pd

# Lê o CSV que está na mesma pasta do notebook.
df = pd.read_csv("halloween_vendas.csv")

# Guarda a categoria original antes de qualquer tratamento.
categoria_original = df["categoria"].astype("string").copy()

# Confirma que o arquivo foi carregado e informa o tamanho da tabela.
print("Arquivo carregado: halloween_vendas.csv")
print("Linhas e colunas:", df.shape)

Arquivo carregado: halloween_vendas.csv
Linhas e colunas: (1500, 14)


#### 1.3) - Conferir as linhas, os nomes das colunas e o total de registros.

In [4]:
# Mostra quantos registros foram lidos.
print("Total de registros:", len(df))

# Lista os nomes das colunas disponíveis.
print("Colunas:", df.columns.tolist())

# Exibe as cinco primeiras linhas para conferir os dados.
display(df.head())

Total de registros: 1500
Colunas: ['pedido_id', 'data_venda', 'produto', 'categoria', 'canal_venda', 'cidade', 'quantidade', 'preco_unitario', 'custo_unitario', 'desconto_pct', 'status_pedido', 'publico', 'tamanho', 'data_entrega']


,pedido_id,data_venda,produto,categoria,canal_venda,cidade,quantidade,preco_unitario,custo_unitario,desconto_pct,status_pedido,publico,tamanho,data_entrega
0,844,2025-11-21 18:02:35,Fantasia de vampiro,Fantasia,Loja física,Curitiba,3,132.5,72.08,15,Cancelado,Adulto,G,NaN
1,378,2024-10-29 18:41:26,Fantasia de bruxa,FANTASIA!!!,Loja física,Maringá,3,110.0,57.00,5,Concluído,Adulto,M,2024-11-03 18:41:26
2,953,2025-09-05 21:16:09,Fantasia de bruxa,Fantasia,Web,Maringá,2,116.6,60.42,5,Concluído,Adulto,M,2025-09-10 21:16:09
3,11,2024-11-08 11:29:44,Mascara assustadora,Acessorio,Loja física,Cascavel,1,35.0,12.00,15,Concluído,Todos,Unico,2024-11-11 11:29:44
4,1335,2025-08-23 11:22:05,Mascara assustadora,Acessorio,Mercado,Curitiba,3,37.1,12.72,0,Concluído,Todos,Unico,2025-08-25 11:22:05


### 1.4) - Preencher categorias vazias com a string "sem categoria".

In [5]:
# Identifica categorias originalmente ausentes ou formadas apenas por espaços.
categorias_vazias = (
    categoria_original.isna()
    | categoria_original.str.strip().eq("").fillna(False)
)

# Conta as ausências antes de preenchê-las.
total_categorias_vazias = int(categorias_vazias.sum())

# Preenche somente as categorias vazias.
df.loc[categorias_vazias, "categoria"] = "sem categoria"

# Informa o total preenchido.
print("Categorias vazias preenchidas:", total_categorias_vazias)

# Mostra o valor original e o valor após o preenchimento.
resultado_preenchimento = df.loc[
    categorias_vazias, ["pedido_id"]
].copy()

resultado_preenchimento["categoria original"] = categoria_original.loc[
    categorias_vazias
]
resultado_preenchimento["categoria após preenchimento"] = df.loc[
    categorias_vazias, "categoria"
]

display(resultado_preenchimento)

Categorias vazias preenchidas: 18


,pedido_id,categoria original,categoria após preenchimento
39,80,<NA>,sem categoria
55,1040,<NA>,sem categoria
157,320,<NA>,sem categoria
233,960,<NA>,sem categoria
240,720,<NA>,sem categoria
371,1120,<NA>,sem categoria
395,1200,<NA>,sem categoria
418,480,<NA>,sem categoria
500,1360,<NA>,sem categoria
777,800,<NA>,sem categoria


### 1.5) - Verificar se quantidade é inteira e maior que zero

In [6]:
# Tenta interpretar a coluna quantidade como numérica.
quantidade_num = pd.to_numeric(df["quantidade"], errors="coerce")

# Considera válida a quantidade numérica, inteira e maior que zero.
quantidade_valida = (
    quantidade_num.notna()
    & quantidade_num.gt(0)
    & quantidade_num.mod(1).eq(0)
)

# Marca como inválidas as quantidades que não atendem a todos os critérios.
quantidade_invalida = ~quantidade_valida

# Exibe quantos valores passaram e quantos falharam na validação.
print("Quantidades válidas:", int(quantidade_valida.sum()))
print("Quantidades inválidas:", int(quantidade_invalida.sum()))

# Mostra os pedidos com quantidade inválida para conferência.
display(
    df.loc[quantidade_invalida, ["pedido_id", "quantidade"]]
)

Quantidades válidas: 1493
Quantidades inválidas: 7


,pedido_id,quantidade
164,630,-1
672,840,-1
845,420,-1
1126,210,-1
1260,1050,-1
1448,1260,-1
1481,1470,-1


### 1.6) - Verificar se preco_unitario está preenchido e não é negativo

In [ ]:
# Tenta interpretar a coluna preco_unitario como numérica.
preco_num = pd.to_numeric(df["preco_unitario"], errors="coerce")

# Considera válido o preço numérico e maior ou igual a zero.
preco_valido = preco_num.notna() & preco_num.ge(0)

# Marca como inválidos os preços que não atendem ao critério.
preco_invalido = ~preco_valido

# Exibe quantos preços passaram e quantos falharam na validação.
print("Preços válidos:", int(preco_valido.sum()))
print("Preços inválidos:", int(preco_invalido.sum()))

# Mostra os pedidos com preço inválido para conferência.
display(
    df.loc[preco_invalido, ["pedido_id", "preco_unitario"]]
)

Preços válidos: 1488
Preços inválidos: 12


,pedido_id,preco_unitario
23,1500,NaN
199,625,NaN
216,1000,NaN
239,1250,NaN
293,500,NaN
503,375,NaN
571,250,NaN
583,125,NaN
1066,875,NaN
1092,750,NaN


### 1.7) - Marcar os registros com problemas

In [8]:
# Guarda as marcações de problema em colunas do DataFrame.
df["problema_quantidade"] = quantidade_invalida
df["problema_preco_unitario"] = preco_invalido

# Marca pedidos com problema em quantidade ou em preço.
df["tem_problema_numerico"] = (
    df["problema_quantidade"] | df["problema_preco_unitario"]
)

# Mostra quantos problemas existem em cada campo e quantos pedidos foram afetados.
print("Problemas em quantidade:", int(df["problema_quantidade"].sum()))
print("Problemas em preço unitário:", int(df["problema_preco_unitario"].sum()))
print("Total de pedidos afetados:", int(df["tem_problema_numerico"].sum()))

# Seleciona os pedidos afetados para mostrar os detalhes.
pedidos_com_problema = df.loc[df["tem_problema_numerico"]]

# Exibe o identificador, os valores e as marcações de problema.
display(
    pedidos_com_problema[
        [
            "pedido_id",
            "quantidade",
            "preco_unitario",
            "problema_quantidade",
            "problema_preco_unitario",
        ]
    ]
)

Problemas em quantidade: 7
Problemas em preço unitário: 12
Total de pedidos afetados: 19


,pedido_id,quantidade,preco_unitario,problema_quantidade,problema_preco_unitario
23,1500,3,NaN,False,True
164,630,-1,22.00,True,False
199,625,2,NaN,False,True
216,1000,1,NaN,False,True
239,1250,1,NaN,False,True
293,500,1,NaN,False,True
503,375,3,NaN,False,True
571,250,4,NaN,False,True
583,125,1,NaN,False,True
672,840,-1,23.32,True,False


### 1.8) - Justificar regra nos comentários e no README.

Categorias vazias ou preenchidas apenas com espaços foram substituídas por “sem categoria”. A quantidade foi considerada válida quando podia ser interpretada como número inteiro maior que zero. O preço unitário foi considerado válido quando estava preenchido, podia ser interpretado como número e era maior ou igual a zero. Registros problemáticos foram marcados, sem inventar ou substituir valores.

### 1.9) - Conte os problemas por campo.

In [9]:
# Conta os registros marcados com problema na quantidade.
total_problemas_quantidade = int(df["problema_quantidade"].sum())

# Conta os registros marcados com problema no preço unitário.
total_problemas_preco = int(df["problema_preco_unitario"].sum())

# Exibe a contagem de problemas de cada campo.
print("Problemas em quantidade:", total_problemas_quantidade)
print("Problemas em preço unitário:", total_problemas_preco)

Problemas em quantidade: 7
Problemas em preço unitário: 12


### 1.10) - Total de pedidos afetados por estes problemas e evitar sobreposição.

In [10]:
# Marca pedidos com problema na quantidade ou no preço unitário.
# Cada pedido será selecionado uma única vez, mesmo se tiver os dois problemas.
pedido_com_problema = (
    df["problema_quantidade"] | df["problema_preco_unitario"]
)

# Seleciona os pedidos que têm pelo menos um problema.
pedidos_afetados = df.loc[pedido_com_problema]

# Exibe o total de pedidos afetados.
print("Total de pedidos afetados:", len(pedidos_afetados))

# Mostra os identificadores e indica qual campo tem problema.
display(
    pedidos_afetados[
        [
            "pedido_id",
            "problema_quantidade",
            "problema_preco_unitario",
        ]
    ]
)

Total de pedidos afetados: 19


,pedido_id,problema_quantidade,problema_preco_unitario
23,1500,False,True
164,630,True,False
199,625,False,True
216,1000,False,True
239,1250,False,True
293,500,False,True
503,375,False,True
571,250,False,True
583,125,False,True
672,840,True,False


## 2 - Padronização de Strings e Regex

   ### 2.1) - Retirar a pontuação indevida com .strip(), .lower() e re.sub()


In [ ]:
import re

# Guarda o conteúdo atual antes da limpeza para poder comparar.
categoria_antes_2_1 = df["categoria"].copy()

# Remove espaços das pontas, converte para minúsculas
# e trata valores ausentes como texto vazio.
categoria_limpa = (
    categoria_antes_2_1.astype("string")
    .str.strip()
    .str.lower()
    .fillna("")
)

# Remove pontuação, preservando letras sem acento e espaços.
categoria_limpa = categoria_limpa.map(
    lambda texto: re.sub(r"[^a-z\s]", "", texto).strip()
)

# Atualiza a coluna com as categorias limpas.
df["categoria"] = categoria_limpa

# Informa quantas categorias diferentes havia antes e depois da limpeza.
print("Categorias diferentes antes da limpeza:", categoria_antes_2_1.nunique())
print("Categorias diferentes depois da limpeza:", df["categoria"].nunique())

# Mostra exemplos comparando o texto original com o resultado.
comparacao_categorias = pd.DataFrame({
    "categoria original": categoria_original,
    "categoria após limpeza": df["categoria"],
})

display(comparacao_categorias.head(10))

Categorias diferentes antes da limpeza: 7
Categorias diferentes depois da limpeza: 3


,categoria original,categoria após limpeza
0,Fantasia,fantasia
1,FANTASIA!!!,fantasia
2,Fantasia,fantasia
3,Acessorio,acessorio
4,Acessorio,acessorio
5,Fantasia,fantasia
6,Fantasia,fantasia
7,Acessorio,acessorio
8,Fantasia,fantasia
9,Fantasia,fantasia


   ### 2.2) - Preencher ausências com ‘sem categoria’.

In [12]:
# Guarda a categoria já limpa antes do preenchimento, para identificar mudanças.
categoria_apos_limpeza = df["categoria"].astype("string").str.strip()

# Marca valores ausentes ou vazios após a limpeza do item 2.1.
vazias_apos_limpeza = (
    categoria_apos_limpeza.isna()
    | categoria_apos_limpeza.eq("").fillna(False)
)

# Conta quantas categorias ficaram vazias após a limpeza.
total_vazias_apos_limpeza = int(vazias_apos_limpeza.sum())

# Preenche somente essas categorias com "sem categoria".
df.loc[vazias_apos_limpeza, "categoria"] = "sem categoria"

# Exibe a contagem e os registros preenchidos.
print("Categorias preenchidas após a limpeza:", total_vazias_apos_limpeza)

resultado_preenchimento_2_2 = df.loc[
    vazias_apos_limpeza, ["pedido_id"]
].copy()

resultado_preenchimento_2_2["categoria original"] = categoria_original.loc[
    vazias_apos_limpeza
]
resultado_preenchimento_2_2["categoria final"] = df.loc[
    vazias_apos_limpeza, "categoria"
]

display(resultado_preenchimento_2_2)

Categorias preenchidas após a limpeza: 0


,pedido_id,categoria original,categoria final


### 2.3) - Contabilista categorias vazias que foram preenchidas e as não vazias alteradas separadamente.

In [13]:
# Prepara a cópia original para comparar com o resultado final.
categoria_original_texto = categoria_original.astype("string")

# Marca as categorias que estavam vazias no CSV original.
vazias_no_original = (
    categoria_original_texto.isna()
    | categoria_original_texto.str.strip().eq("").fillna(False)
)

# Compara as categorias finais com as originais.
categoria_final = df["categoria"].astype("string")

# Marca como alteradas somente as categorias que não estavam vazias originalmente.
nao_vazias_alteradas = (
    ~vazias_no_original
    & categoria_final.ne(categoria_original_texto).fillna(False)
)

# Conta cada grupo separadamente.
total_vazias_preenchidas = int(vazias_no_original.sum())
total_nao_vazias_alteradas = int(nao_vazias_alteradas.sum())

# Exibe as duas contagens.
print("Categorias vazias preenchidas:", total_vazias_preenchidas)
print("Categorias não vazias alteradas:", total_nao_vazias_alteradas)

# Mostra exemplos das categorias não vazias que foram alteradas.
exemplos_alteracoes = pd.DataFrame({
    "categoria original": categoria_original_texto[nao_vazias_alteradas],
    "categoria final": categoria_final[nao_vazias_alteradas],
})

display(exemplos_alteracoes.head(10))

Categorias vazias preenchidas: 18
Categorias não vazias alteradas: 1482


,categoria original,categoria final
0,Fantasia,fantasia
1,FANTASIA!!!,fantasia
2,Fantasia,fantasia
3,Acessorio,acessorio
4,Acessorio,acessorio
5,Fantasia,fantasia
6,Fantasia,fantasia
7,Acessorio,acessorio
8,Fantasia,fantasia
9,Fantasia,fantasia


### 2.4) - Padronizar canal de venda para Mercado, Loja Física e Web.

In [14]:
# Guarda os valores atuais para conferir o resultado da padronização.
canal_original = df["canal_venda"].copy()

# Remove espaços das pontas e converte os canais para minúsculas.
canal_normalizado = (
    df["canal_venda"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Relaciona cada forma normalizada ao nome padronizado.
mapa_canais = {
    "mercado": "Mercado",
    "loja física": "Loja física",
    "web": "Web",
}

# Aplica o mapeamento aos canais.
canal_padronizado = canal_normalizado.map(mapa_canais)

# Identifica e informa canais ausentes ou não reconhecidos.
canais_invalidos = canal_normalizado[canal_padronizado.isna()]
print("Registros com canal não reconhecido:", len(canais_invalidos))
print("Valores não reconhecidos:", canais_invalidos.fillna("<ausente>").unique().tolist())

# Interrompe com uma mensagem se houver canais que não estejam no mapa.
if not canais_invalidos.empty:
    raise ValueError("Revise os canais não reconhecidos antes de continuar.")

# Atualiza a coluna com os nomes padronizados.
df["canal_venda"] = canal_padronizado

# Conta e exibe os pedidos por canal.
contagem_canais = (
    df["canal_venda"]
    .value_counts()
    .rename_axis("Canal de venda")
    .reset_index(name="Total de pedidos")
)

print("Pedidos por canal padronizado:")
display(contagem_canais)

# Mostra exemplos antes e depois para conferir a transformação.
comparacao_canais = pd.DataFrame({
    "canal original": canal_original,
    "canal padronizado": df["canal_venda"],
})

display(comparacao_canais.head(10))

Registros com canal não reconhecido: 0
Valores não reconhecidos: []
Pedidos por canal padronizado:


,Canal de venda,Total de pedidos
0,Web,666
1,Loja física,551
2,Mercado,283


,canal original,canal padronizado
0,Loja física,Loja física
1,Loja física,Loja física
2,Web,Web
3,Loja física,Loja física
4,Mercado,Mercado
5,Loja física,Loja física
6,Web,Web
7,Loja física,Loja física
8,Web,Web
9,Loja física,Loja física


# 3 - Lógica de Regra de Negócio (Filtros e Validação)

### 3.1) - Verificar se todos pedidos com data vazia foram cancelados.

In [15]:
# Prepara a data de entrega como texto e remove espaços das pontas.
data_entrega_texto = df["data_entrega"].astype("string").str.strip()

# Marca valores ausentes ou textos vazios como sem data de entrega.
data_vazia = (
    df["data_entrega"].isna()
    | data_entrega_texto.eq("").fillna(False)
)

# Prepara o status para comparar sem diferença entre espaços ou maiúsculas.
status_normalizado = (
    df["status_pedido"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

# Marca como True os pedidos cujo status é "Cancelado".
status_cancelado = status_normalizado.eq("cancelado").fillna(False)

# Seleciona todos os pedidos sem data de entrega e guarda uma cópia para análise.
pedidos_sem_entrega = df.loc[data_vazia].copy()

# Acrescenta uma coluna que informa se cada pedido sem data foi cancelado.
pedidos_sem_entrega["status_cancelado"] = status_cancelado.loc[data_vazia]

# Informa quantos pedidos sem data foram encontrados.
print("Pedidos sem data de entrega:", len(pedidos_sem_entrega))

# Exibe todos os pedidos sem data e indica se o status é Cancelado.
display(
    pedidos_sem_entrega[
        ["pedido_id", "status_pedido", "data_entrega", "status_cancelado"]
    ]
)

Pedidos sem data de entrega: 143


,pedido_id,status_pedido,data_entrega,status_cancelado
0,844,Cancelado,NaN,True
8,186,Cancelado,NaN,True
21,662,Cancelado,NaN,True
49,1170,Concluído,NaN,False
59,896,Cancelado,NaN,True
...,...,...,...,...
1448,1260,Concluído,NaN,False
1453,298,Cancelado,NaN,True
1458,102,Cancelado,NaN,True
1480,1490,Cancelado,NaN,True


### 3.2) - Contar pedidos sem entrega e cancelados, e pedidos sem entrega com outro status.

In [16]:
# Separa os pedidos sem data de entrega que foram cancelados.
pedidos_sem_entrega_cancelados = pedidos_sem_entrega.loc[
    pedidos_sem_entrega["status_cancelado"]
]

# Separa os pedidos sem data de entrega com outro status.
pedidos_sem_entrega_outro_status = pedidos_sem_entrega.loc[
    ~pedidos_sem_entrega["status_cancelado"]
]

# Exibe a quantidade de pedidos em cada grupo.
print("Sem entrega e cancelados:", len(pedidos_sem_entrega_cancelados))
print("Sem entrega com outro status:", len(pedidos_sem_entrega_outro_status))

# Confere se os dois grupos juntos correspondem ao total sem data.
print(
    "Total conferido:",
    len(pedidos_sem_entrega_cancelados)
    + len(pedidos_sem_entrega_outro_status)
)

# Mostra quais outros status aparecem e quantos pedidos há de cada um.
print("Distribuição dos outros status:")
display(
    pedidos_sem_entrega_outro_status["status_pedido"]
    .value_counts(dropna=False)
    .rename_axis("Status")
    .reset_index(name="Quantidade")
)

Sem entrega e cancelados: 109
Sem entrega com outro status: 34
Total conferido: 143
Distribuição dos outros status:


,Status,Quantidade
0,Concluído,34


### 3.3) - Mostra pelo menos 3 exemplos de ‘pedidos sem entrega e com outro status’.

In [17]:
# Informa o que será exibido.
print("Exemplos de pedidos sem entrega com outro status:")

# Escolhe as colunas que ajudam a identificar e conferir os pedidos.
colunas_exibidas = [
    "pedido_id",
    "status_pedido",
    "data_entrega",
]

# Mostra até três exemplos do grupo separado no item 3.2.
display(
    pedidos_sem_entrega_outro_status[colunas_exibidas].head(3)
)

Exemplos de pedidos sem entrega com outro status:


,pedido_id,status_pedido,data_entrega
49,1170,Concluído,NaN
117,990,Concluído,NaN
152,740,Concluído,NaN


### 3.4) - Todo pedido sem entrega foi cancelado? 

In [18]:
# Verifica se há pedidos sem data de entrega.
if pedidos_sem_entrega.empty:
    print("Não há pedidos sem data de entrega para verificar.")

# Se não há pedidos com outro status, todos os pedidos sem entrega foram cancelados.
elif pedidos_sem_entrega_outro_status.empty:
    print("Sim: todos os pedidos sem data de entrega foram cancelados.")

# Se existe algum pedido com outro status, a hipótese não é verdadeira.
else:
    print("Não: há pedidos sem data de entrega com outro status.")

Não: há pedidos sem data de entrega com outro status.


# 4 - Formatação Temporal (Datetime)

### 4.1 - Preservar o texto original: manter uma cópia do texto original da data.


In [19]:
# Cria uma cópia da coluna data_venda para preservar o texto original.
df["data_venda_original"] = df["data_venda"].copy()

# Exibe as duas colunas para conferir que o texto foi preservado.
display(df[["data_venda", "data_venda_original"]].head())

,data_venda,data_venda_original
0,2025-11-21 18:02:35,2025-11-21 18:02:35
1,2024-10-29 18:41:26,2024-10-29 18:41:26
2,2025-09-05 21:16:09,2025-09-05 21:16:09
3,2024-11-08 11:29:44,2024-11-08 11:29:44
4,2025-08-23 11:22:05,2025-08-23 11:22:05


### 4.2 - Preservar o texto original: Manter o registo original do texto.


In [20]:
from datetime import datetime

formato_origem = "%Y-%m-%d %H:%M:%S"

def converter_data(texto):
    if pd.isna(texto) or str(texto).strip() == "":
        return None

    try:
        return datetime.strptime(str(texto).strip(), formato_origem)
    except ValueError:
        return None

df["data_venda_convertida"] = df["data_venda_original"].map(converter_data)

# Mostra a data original e o resultado da conversão
display(
    df[["data_venda_original", "data_venda_convertida"]].head(10)
)

# Conta e mostra datas preenchidas que não puderam ser convertidas
texto_data = df["data_venda_original"].astype("string").str.strip()
datas_invalidas = (
    texto_data.notna()
    & texto_data.ne("")
    & df["data_venda_convertida"].isna()
)

print("Total de datas inválidas:", int(datas_invalidas.sum()))
display(df.loc[datas_invalidas, ["data_venda_original", "data_venda_convertida"]])

,data_venda_original,data_venda_convertida
0,2025-11-21 18:02:35,2025-11-21 18:02:35
1,2024-10-29 18:41:26,2024-10-29 18:41:26
2,2025-09-05 21:16:09,2025-09-05 21:16:09
3,2024-11-08 11:29:44,2024-11-08 11:29:44
4,2025-08-23 11:22:05,2025-08-23 11:22:05
5,2024-10-01 15:00:11,2024-10-01 15:00:11
6,2025-08-26 19:49:13,2025-08-26 19:49:13
7,2024-10-12 14:13:18,2024-10-12 14:13:18
8,2024-08-17 22:17:18,2024-08-17 22:17:18
9,2025-10-11 19:01:38,2025-10-11 19:01:38


Total de datas inválidas: 4


,data_venda_original,data_venda_convertida
152,data_invalida,NaT
255,data_invalida,NaT
714,data_invalida,NaT
884,data_invalida,NaT


### 4.3 - Extrair componentes da data: Isolar os campos de ano, mês, dia e hora.

In [21]:
# Garante que a coluna esteja no formato de data e hora do Pandas.
datas_convertidas = pd.to_datetime(
    df["data_venda_convertida"],
    errors="coerce"
)

# Extrai cada componente para uma coluna própria.
df["ano"] = datas_convertidas.dt.year
df["mes"] = datas_convertidas.dt.month
df["dia"] = datas_convertidas.dt.day
df["hora"] = datas_convertidas.dt.hour

# Exibe os valores para conferência.
display(
    df[
        [
            "data_venda_original",
            "data_venda_convertida",
            "ano",
            "mes",
            "dia",
            "hora",
        ]
    ].head()
)

,data_venda_original,data_venda_convertida,ano,mes,dia,hora
0,2025-11-21 18:02:35,2025-11-21 18:02:35,2025.0,11.0,21.0,18.0
1,2024-10-29 18:41:26,2024-10-29 18:41:26,2024.0,10.0,29.0,18.0
2,2025-09-05 21:16:09,2025-09-05 21:16:09,2025.0,9.0,5.0,21.0
3,2024-11-08 11:29:44,2024-11-08 11:29:44,2024.0,11.0,8.0,11.0
4,2025-08-23 11:22:05,2025-08-23 11:22:05,2025.0,8.0,23.0,11.0


### 4.4 - Formatar para padrão brasileiro utilizando strftime(), com o formato "12/10/2025".


In [22]:
# Formata a data como dia/mês/ano.
df["data_venda_br"] = datas_convertidas.dt.strftime("%d/%m/%Y")

# Exibe o texto original e a data formatada para conferência.
display(
    df[
        [
            "data_venda_original",
            "data_venda_br",
        ]
    ].head()
)

,data_venda_original,data_venda_br
0,2025-11-21 18:02:35,21/11/2025
1,2024-10-29 18:41:26,29/10/2024
2,2025-09-05 21:16:09,05/09/2025
3,2024-11-08 11:29:44,08/11/2024
4,2025-08-23 11:22:05,23/08/2025


### 4.5 - Contar e marcar datas vazias ou inválidas

In [23]:
# Prepara o texto original para verificar se está vazio.
texto_data = df["data_venda_original"].astype("string").str.strip()

# Marca datas ausentes ou que contêm apenas espaços.
data_vazia = (
    texto_data.isna()
    | texto_data.eq("").fillna(False)
)

# Marca datas preenchidas que não foram convertidas.
data_invalida = (
    ~data_vazia
    & df["data_venda_convertida"].isna()
)

# Registra as marcações em colunas novas.
df["data_venda_vazia"] = data_vazia
df["data_venda_invalida"] = data_invalida
df["data_venda_problematica"] = data_vazia | data_invalida

# Conta os casos encontrados.
total_vazias = int(data_vazia.sum())
total_invalidas = int(data_invalida.sum())
total_problematicas = int(df["data_venda_problematica"].sum())

print("Datas vazias:", total_vazias)
print("Datas inválidas:", total_invalidas)
print("Total de datas vazias ou inválidas:", total_problematicas)

Datas vazias: 0
Datas inválidas: 4
Total de datas vazias ou inválidas: 4


### 4.6 — Não preencher com a data atual

In [24]:
# Seleciona as linhas que foram marcadas como vazias ou inválidas no item 4.5.
linhas_com_problema = df["data_venda_problematica"]

# Conta quantas dessas linhas continuam sem uma data convertida.
total_sem_data = int(
    df.loc[linhas_com_problema, "data_venda_convertida"].isna().sum()
)

print("Registros problemáticos mantidos sem data:", total_sem_data)

# Exibe exemplos para conferir o texto original e o valor convertido.
display(
    df.loc[
        linhas_com_problema,
        ["data_venda_original", "data_venda_convertida"]
    ].head()
)

Registros problemáticos mantidos sem data: 4


,data_venda_original,data_venda_convertida
152,data_invalida,NaT
255,data_invalida,NaT
714,data_invalida,NaT
884,data_invalida,NaT


# 5 - Relatório de Status Manual e Exploração

### 5.1 - Exibir um resumo

In [25]:
# Prepara os status para comparação, ignorando espaços e maiúsculas/minúsculas.
status = df["status_pedido"].astype("string").str.strip().str.casefold()

# Identifica pedidos sem data de entrega.
texto_entrega = df["data_entrega"].astype("string").str.strip()
sem_entrega = (
    df["data_entrega"].isna()
    | texto_entrega.eq("").fillna(False)
)

# Separa pedidos sem entrega entre cancelados e com outro status.
sem_entrega_cancelados = sem_entrega & status.eq("cancelado").fillna(False)
sem_entrega_outro_status = sem_entrega & ~status.eq("cancelado").fillna(False)

# Conta os pedidos cancelados, independentemente da data de entrega.
total_cancelados = int(status.eq("cancelado").fillna(False).sum())

# Monta uma tabela com os principais resultados.
resumo = pd.DataFrame({
    "Indicador": [
        "Total de linhas lidas",
        "Categorias vazias preenchidas",
        "Categorias não vazias alteradas",
        "Registros com problemas numéricos",
        "Datas de venda inválidas",
        "Pedidos cancelados",
        "Pedidos sem entrega e cancelados",
        "Pedidos sem entrega com outro status",
    ],
    "Quantidade": [
        len(df),
        total_vazias_preenchidas,
        total_nao_vazias_alteradas,
        int(df["tem_problema_numerico"].sum()),
        int(df["data_venda_invalida"].sum()),
        total_cancelados,
        int(sem_entrega_cancelados.sum()),
        int(sem_entrega_outro_status.sum()),
    ],
})

display(resumo)

,Indicador,Quantidade
0,Total de linhas lidas,1500
1,Categorias vazias preenchidas,18
2,Categorias não vazias alteradas,1482
3,Registros com problemas numéricos,19
4,Datas de venda inválidas,4
5,Pedidos cancelados,109
6,Pedidos sem entrega e cancelados,109
7,Pedidos sem entrega com outro status,34


### 5.2 - Contar pedidos por canal de venda

In [26]:
# Lista os canais que serão considerados reconhecidos.
canais_reconhecidos = ["Mercado", "Loja física", "Web"]

# Remove espaços das pontas antes de conferir cada canal.
canal_texto = df["canal_venda"].astype("string").str.strip()

# Marca os registros cujo canal está na lista de canais reconhecidos.
canal_reconhecido = canal_texto.isin(canais_reconhecidos)

# Conta pedidos por canal reconhecido e mostra também canais com zero pedidos.
contagem_por_canal = (
    canal_texto[canal_reconhecido]
    .value_counts()
    .reindex(canais_reconhecidos, fill_value=0)
    .rename_axis("Canal de venda")
    .reset_index(name="Total de pedidos")
)

# Conta e lista os registros excluídos por terem canal não reconhecido.
total_excluidos = int((~canal_reconhecido).sum())
canais_excluidos = canal_texto[~canal_reconhecido].fillna("<ausente>").unique()

print("Pedidos por canal reconhecido:")
display(contagem_por_canal)

print("Registros excluídos por canal não reconhecido:", total_excluidos)
print("Canais encontrados entre os registros excluídos:", canais_excluidos.tolist())

Pedidos por canal reconhecido:


,Canal de venda,Total de pedidos
0,Mercado,283
1,Loja física,551
2,Web,666


Registros excluídos por canal não reconhecido: 0
Canais encontrados entre os registros excluídos: []


### 5.3 - Contar pedidos por ano e mês

In [27]:
# Garante que os valores estejam no formato de data do Pandas.
datas_venda = pd.to_datetime(
    df["data_venda_convertida"],
    errors="coerce"
)

# Marca quais registros têm uma data válida.
data_valida = datas_venda.notna()

# Separa o ano e o mês como um único período, por exemplo: 2024-10.
ano_mes = datas_venda[data_valida].dt.to_period("M")

# Conta os pedidos em cada período e organiza do mais antigo ao mais recente.
contagem_por_periodo = (
    ano_mes.value_counts()
    .sort_index()
    .rename_axis("Ano e mês")
    .reset_index(name="Total de pedidos")
)

# Conta os registros excluídos por não terem data válida.
total_excluidos = int((~data_valida).sum())

print("Pedidos registrados por ano e mês:")
display(contagem_por_periodo)

print("Registros excluídos por data de venda ausente ou inválida:", total_excluidos)

Pedidos registrados por ano e mês:


,Ano e mês,Total de pedidos
0,2024-08,77
1,2024-09,104
2,2024-10,393
3,2024-11,174
4,2025-08,62
5,2025-09,94
6,2025-10,410
7,2025-11,182


Registros excluídos por data de venda ausente ou inválida: 4


### 5.4 - Nas contagens por canal e por período

In [28]:
# Conta os status nos registros com canal reconhecido.
# dropna=False inclui também pedidos cujo status esteja vazio.
status_por_canal = (
    df.loc[canal_reconhecido, "status_pedido"]
    .value_counts(dropna=False)
    .rename_axis("Status")
    .reset_index(name="Total de pedidos")
)

# Conta os status nos registros com data de venda válida.
status_por_periodo = (
    df.loc[data_valida, "status_pedido"]
    .value_counts(dropna=False)
    .rename_axis("Status")
    .reset_index(name="Total de pedidos")
)

# Conta quantos registros entraram e quantos foram excluídos em cada análise.
incluidos_canal = int(canal_reconhecido.sum())
excluidos_canal = int((~canal_reconhecido).sum())

incluidos_periodo = int(data_valida.sum())
excluidos_periodo = int((~data_valida).sum())

print("Por canal — todos os status nos registros com canal reconhecido:")
display(status_por_canal)
print("Registros incluídos:", incluidos_canal)
print("Registros excluídos por canal não reconhecido:", excluidos_canal)

print("\nPor período — todos os status nos registros com data válida:")
display(status_por_periodo)
print("Registros incluídos:", incluidos_periodo)
print("Registros excluídos por data ausente ou inválida:", excluidos_periodo)

Por canal — todos os status nos registros com canal reconhecido:


,Status,Total de pedidos
0,Concluído,1345
1,Cancelado,109
2,Devolvido,46


Registros incluídos: 1500
Registros excluídos por canal não reconhecido: 0

Por período — todos os status nos registros com data válida:


,Status,Total de pedidos
0,Concluído,1341
1,Cancelado,109
2,Devolvido,46


Registros incluídos: 1496
Registros excluídos por data ausente ou inválida: 4


### 5.5 - Apresentar duas descobertas

In [29]:
# Descoberta por canal: encontra o canal com mais pedidos.
indice_maior_canal = contagem_por_canal["Total de pedidos"].idxmax()
maior_canal = contagem_por_canal.loc[indice_maior_canal]

total_canais_contados = contagem_por_canal["Total de pedidos"].sum()

if total_canais_contados > 0:
    percentual_maior_canal = (
        maior_canal["Total de pedidos"] / total_canais_contados
    ) * 100

    print(
        f"Descoberta por canal: {maior_canal['Canal de venda']} "
        f"teve mais pedidos, com {maior_canal['Total de pedidos']} "
        f"de {total_canais_contados} pedidos contados "
        f"({percentual_maior_canal:.1f}%)."
    )
    print(
        "Interpretação: esse foi o canal com maior número de pedidos "
        "registrados entre os canais reconhecidos."
    )
    print(
        "Limitação: a contagem inclui todos os status e não representa "
        "receita nem somente vendas concluídas; canais não reconhecidos "
        "foram excluídos."
    )
else:
    print("Descoberta por canal: não há pedidos com canal reconhecido para comparar.")


# Descoberta por período: encontra o ano e mês com mais pedidos.
if not contagem_por_periodo.empty:
    indice_maior_periodo = contagem_por_periodo["Total de pedidos"].idxmax()
    maior_periodo = contagem_por_periodo.loc[indice_maior_periodo]

    total_periodos_contados = contagem_por_periodo["Total de pedidos"].sum()
    percentual_maior_periodo = (
        maior_periodo["Total de pedidos"] / total_periodos_contados
    ) * 100

    print(
        f"\nDescoberta por período: {maior_periodo['Ano e mês']} "
        f"teve mais pedidos, com {maior_periodo['Total de pedidos']} "
        f"de {total_periodos_contados} pedidos contados "
        f"({percentual_maior_periodo:.1f}%)."
    )
    print(
        "Interpretação: esse foi o período com maior número de pedidos "
        "registrados entre os que tinham data de venda válida."
    )
    print(
        "Limitação: a contagem inclui todos os status e não representa "
        "receita nem somente vendas concluídas; datas ausentes ou inválidas "
        "foram excluídas."
    )
else:
    print("\nDescoberta por período: não há datas válidas para comparar.")

Descoberta por canal: Web teve mais pedidos, com 666 de 1500 pedidos contados (44.4%).
Interpretação: esse foi o canal com maior número de pedidos registrados entre os canais reconhecidos.
Limitação: a contagem inclui todos os status e não representa receita nem somente vendas concluídas; canais não reconhecidos foram excluídos.

Descoberta por período: 2025-10 teve mais pedidos, com 410 de 1496 pedidos contados (27.4%).
Interpretação: esse foi o período com maior número de pedidos registrados entre os que tinham data de venda válida.
Limitação: a contagem inclui todos os status e não representa receita nem somente vendas concluídas; datas ausentes ou inválidas foram excluídas.


### 5.6 - Esclarecer o que as contagens representam

In [30]:
print("Total de linhas analisadas:", len(df))

with open("report.md", "r", encoding="utf-8") as arquivo:
    print(arquivo.read())

Total de linhas analisadas: 1500
# Relatório e insights

## Objetivo e tratamento dos dados

Analisei 1.500 pedidos do arquivo `halloween_vendas.csv` para verificar a qualidade dos dados e observar sua distribuição por canal e período. Preservei os valores originais quando necessário e fiz os tratamentos no DataFrame, sem alterar o CSV.

Na coluna `categoria`, removi espaços excedentes, converti o texto para minúsculas e usei `re.sub` para retirar pontuação, mantendo letras sem acento e espaços. Categorias vazias ou que ficaram vazias após a limpeza receberam o valor “sem categoria”. No arquivo original, 18 categorias estavam vazias e 1.482 categorias não vazias foram alteradas.

Também validei os campos numéricos: 7 registros tinham problemas em `quantidade` e 12 em `preco_unitario`. Ao combinar as marcações, encontrei 19 pedidos afetados, sem contar um pedido mais de uma vez. Quatro datas de venda eram inválidas e permaneceram sem preenchimento. Não inventei valores para campos ausen

### 6.1 - Exibir um resumo 'report.md' com o total de linhas analisados.

In [31]:
from pathlib import Path

texto_relatorio = Path("report.md").read_text(encoding="utf-8")
total_caracteres = len(texto_relatorio)

print("Caracteres no relatório:", total_caracteres)

if 1500 <= total_caracteres <= 3000:
    print("O relatório está dentro do limite.")
else:
    print("O relatório está fora do limite.")

Caracteres no relatório: 2391
O relatório está dentro do limite.
